# 02. LOT별로 Z3·Z4·Z5 온도와 출력을 겹쳐 보기

[이슈 #71](https://github.com/posco-3bungi-pdm/pdm-project/issues/71) · 예측 대상은 `TC-Z3`, `OP-Z3`, `TC-Z4`, `OP-Z4`, `TC-Z5`, `OP-Z5`, `CP-Z4` 총 7개임.

**먼저 온도 3개를 겹쳐 보고, 출력 3개를 겹쳐 봄. 그다음 같은 구역의 온도·출력을 겹쳐 봄.**

- 가로축: **2024년 실제 측정 날짜·시각**임. 예를 들어 `01/02 13:26`은 1월 2일 오후 1시 26분임.
- 진한 선: 해당 시각부터 1분 동안의 평균임. 옅은 색: 같은 1분 안의 최소~최대임.
- 온도는 ℃, 출력은 %로 표시함. 두 단위는 데이터사전의 가정임. `CP-Z4`의 단위는 확인되지 않아 기록값으로 표시함.
- 관측이 없는 분은 비워 둠. 원자료와 출력 0을 지우지 않음. 관측 30개 미만을 일괄 삭제하던 과거 임시 기준은 적용하지 않음.
- 현재 자료를 설명하는 그림임. 모델 학습·시험 분할이나 이상 정답을 만드는 과정은 아님.

`VIEW_LOT`만 바꾸면 화면에 볼 LOT가 달라짐. 전체 14개 LOT의 그림은 매번 같은 파일명으로 저장되므로 중복 파일이 쌓이지 않음. 이전 파생변수 노트북을 먼저 실행할 필요가 없음.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager
from IPython.display import display

# 저장소, notebooks, 현재 하위 폴더 중 어디서 실행해도 같은 원자료를 읽기 위함
ROOT = Path.cwd()
if ROOT.name == "타깃_채널_분석_namelesspark":
    ROOT = ROOT.parent.parent
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
if not (DATA / "T-CR1-CAL01_온도.csv").exists():
    DATA = DATA / "raw"
FIGURES = ROOT / "figures" / "타깃_채널_비교_namelesspark"

# 설치된 한글 폰트를 쓰기 위함
fonts = {font.name for font in font_manager.fontManager.ttflist}
for font in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if font in fonts:
        plt.rcParams["font.family"] = font
        break
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False

TARGETS = ["TC-Z3", "OP-Z3", "TC-Z4", "OP-Z4", "TC-Z5", "OP-Z5", "CP-Z4"]
raw = pd.read_csv(DATA / "T-CR1-CAL01_온도.csv", encoding="utf-8-sig", parse_dates=["MEAS_DT"])
events = pd.read_csv(DATA / "G-02_조업이벤트.csv", encoding="utf-8-sig", parse_dates=["EVT_DT"])
raw = raw.sort_values(["LOT_NO", "MEAS_DT"])

# 이 자료는 LOT 첫 관측과 강종변경 시각이 일치함. 해당 시각으로 강종을 연결함
grade_events = events.loc[events.PLANT_CD.eq("CR1") & events.EVT_TYPE.eq("강종변경")]
lots = raw.groupby("LOT_NO").agg(시작=("MEAS_DT", "min"), 끝=("MEAS_DT", "max"))
lots["강종"] = lots.시작.map(grade_events.set_index("EVT_DT").AFT_VAL)

# LOT마다 따로 집계하므로 서로 다른 LOT 사이의 빈 날짜를 메우지 않음
# OP의 0은 실제 출력임. 결측으로 바꾸지 않음
means, lows, highs, counts = {}, {}, {}, {}
for lot, rows in raw.groupby("LOT_NO"):
    minutes = rows.set_index("MEAS_DT")[TARGETS].resample("1min")
    means[lot] = minutes.mean()
    lows[lot] = minutes.min()
    highs[lot] = minutes.max()
    counts[lot] = minutes.count()

# 아래 2개는 팀에서 확정한 해석임. 전체 LOT를 고장으로 표시하는 뜻은 아님
operation = {240058: "정기수리를 위한 정지", 240061: "정기수리 후 재가동"}


## 1. LOT의 강종과 관측 수 확인하기

한 분의 평균이라도 그 안의 실제 관측 수는 다를 수 있음. 마지막 분이 일부만 기록됐을 때도 평균은 남겨 두고 관측 수를 함께 확인함.


In [ ]:
display(lots)
quality = []
for lot, count in counts.items():
    quality.append({
        "LOT": lot,
        "강종": lots.loc[lot, "강종"],
        "전체 분 수": len(count),
        "센서별 관측 수의 최솟값": count.min().min(),
        "센서별 관측 수의 최댓값": count.max().max(),
        "관측이 하나도 없는 센서·분 수": count.eq(0).sum().sum(),
    })
display(pd.DataFrame(quality))


## 2. 온도끼리 → 출력끼리 → 같은 구역의 온도·출력 보기

먼저 위아래 두 그림에서 Z3·Z4·Z5의 선이 함께 움직이는지 보기. 다음 세 그림에서는 같은 구역의 온도·출력 변화 시점을 보기.

**온도선과 출력선의 교차 위치는 의미가 없음. 왼쪽과 오른쪽 축이 다르므로 변화한 시점을 비교하기 위함임.** 같은 LOT의 Z3·Z4·Z5 온도축 범위는 같음. 다른 LOT끼리는 다음 노트북에서 비교함.

마지막 CP 그림도 같은 실제 날짜·시각으로 읽음. 옅은 색의 폭이 넓으면 해당 1분 안에서 값이 크게 움직였다는 뜻임.


In [ ]:
# 화면에는 이 LOT만 표시함. 다른 LOT도 파일로는 모두 저장함
VIEW_LOT = 240001
ZONE_COLORS = {3: "#2B70B2", 4: "#D17A25", 5: "#448A65"}

# 같은 LOT의 세 구역을 같은 온도 범위에서 보기 위함
for lot, mean in means.items():
    low, high = lows[lot], highs[lot]
    time = mean.index
    grade = lots.loc[lot, "강종"]
    note = operation.get(lot, "")
    title = f"LOT {lot} · {grade}" + (f" · {note}" if note else "")
    tc_min = low[["TC-Z3", "TC-Z4", "TC-Z5"]].min().min() - 1
    tc_max = high[["TC-Z3", "TC-Z4", "TC-Z5"]].max().max() + 1

    # 온도끼리, 출력끼리 겹쳐서 구역 차이를 보기 위함
    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, layout="constrained")
    for ax, prefix, unit in zip(axes, ["TC", "OP"], ["온도 (℃)", "출력 (%)"]):
        for zone, color in ZONE_COLORS.items():
            sensor = f"{prefix}-Z{zone}"
            ax.plot(time, mean[sensor], color=color, linewidth=1, label=sensor)
            ax.fill_between(time, low[sensor], high[sensor], color=color, alpha=0.10)
        ax.set_ylabel(unit)
        ax.legend(loc="upper left", ncol=3)
        ax.grid(alpha=0.15)
    axes[0].set_ylim(tc_min, tc_max)
    axes[1].set_ylim(0, 105)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
    axes[1].set_xlabel("2024년 실제 측정 날짜·시각 · 선: 1분 평균 · 옅은 색: 그 분의 최소~최대")
    fig.suptitle(title + " | Z3·Z4·Z5 비교")
    output = FIGURES / "구역_간_비교" / "LOT별"
    output.mkdir(parents=True, exist_ok=True)
    fig.savefig(output / f"{lot}_{grade}_온도_출력_비교.png", dpi=140)
    if lot == VIEW_LOT:
        plt.show()
    plt.close(fig)

    # 온도와 출력은 단위가 다름. 왼쪽 온도축과 오른쪽 출력축으로 구분함
    for zone in [3, 4, 5]:
        tc, op = f"TC-Z{zone}", f"OP-Z{zone}"
        fig, ax = plt.subplots(figsize=(13, 4), layout="constrained")
        right = ax.twinx()
        ax.plot(time, mean[tc], color="#2B70B2", linewidth=1, label=tc)
        ax.fill_between(time, low[tc], high[tc], color="#2B70B2", alpha=0.13)
        right.plot(time, mean[op], color="#D17A25", linewidth=1, label=op)
        right.fill_between(time, low[op], high[op], color="#D17A25", alpha=0.10)
        ax.set_ylim(tc_min, tc_max)
        right.set_ylim(0, 105)
        ax.set_ylabel("온도 (℃)", color="#2B70B2")
        right.set_ylabel("출력 (%)", color="#D17A25")
        ax.legend(loc="upper left")
        right.legend(loc="upper right")
        ax.grid(alpha=0.15)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
        ax.set_xlabel("2024년 실제 측정 날짜·시각 · 선: 1분 평균 · 옅은 색: 그 분의 최소~최대")
        ax.set_title(title + f" | Z{zone} 온도·출력")
        output = FIGURES / f"Z{zone}_온도_출력" / "LOT별"
        output.mkdir(parents=True, exist_ok=True)
        fig.savefig(output / f"{lot}_{grade}_Z{zone}_온도_출력.png", dpi=140)
        if lot == VIEW_LOT:
            plt.show()
        plt.close(fig)

    # CP는 온도·출력과 단위가 달라 별도 그림으로 같은 시각을 확인함
    fig, ax = plt.subplots(figsize=(13, 3.5), layout="constrained")
    ax.plot(time, mean["CP-Z4"], color="#7559A6", linewidth=1, label="CP-Z4")
    ax.fill_between(time, low["CP-Z4"], high["CP-Z4"], color="#7559A6", alpha=0.15)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
    ax.set_ylabel("CP-Z4 기록값 (단위 미확인)")
    ax.set_xlabel("2024년 실제 측정 날짜·시각 · 선: 1분 평균 · 옅은 색: 그 분의 최소~최대")
    ax.set_title(title + " | CP-Z4")
    ax.grid(alpha=0.15)
    ax.legend()
    output = FIGURES / "CP_기록값" / "LOT별"
    output.mkdir(parents=True, exist_ok=True)
    fig.savefig(output / f"{lot}_{grade}_CP_Z4.png", dpi=140)
    if lot == VIEW_LOT:
        plt.show()
    plt.close(fig)


## 3. LOT별로 앞서 확인한 내용 이어서 보기

앞선 상세 검토의 관찰을 보존한 표임. 아래 그림만으로 고장 원인이나 모델 성능이 확정된 것은 아님. 기존 학습 보류 시각은 아래 접힌 설명에서 볼 수 있음.

| LOT · 강종 | 7개 채널에서 확인한 변화와 현재 판단 |
| --- | --- |
| 240001 · SGCC | `TC-Z3`가 오른 시기에 `OP-Z3`가 낮아지고, 뒤이어 온도가 내려갈 때 출력이 높아졌으며 Z4에서도 온도 상승·출력 감소가 보였음. 제어 반응으로 해석할 수 있지만 원인은 확정하지 않았고, SGCC는 이 LOT 하나뿐이므로 LOT 전체를 빼거나 이 범위를 정상 기준으로 확정하지 않음. |
| 240024 · SPFH590 | `CP-Z4`가 짧게 낮아진 변화는 여러 원자료와 1분 평균에 남고, 가까운 시각 Z5에서는 온도 상승·출력 감소가 보였음. 특정 고장을 확정하지 않았으며 기존안에서는 `CP-Z4`의 03/08 04:59~05:00 두 분만 학습에서 보류했음. |
| 240037 · SPCE | `CP-Z4`가 크게 오른 뒤 약 0.45로 돌아왔고, 가까운 시각 Z4·Z5 온도가 오르면서 출력이 줄었음. 서로 함께 변했다는 사실만으로 CP를 원인으로 정하지 않았으며, 기존안은 세 차례의 짧은 CP 이탈 총 7분만 보류했음. |
| 240038 · SPCEN | `TC-Z4`가 비슷한 범위여도 `OP-Z4`는 114의 후반 하강 전보다 높았지만 CP 중앙값은 비슷했고, 이것만으로 SPCEN의 필수 출력이나 설비 고장을 확정하지 못했음. `OP-Z4` 전체 제외는 철회해 유지하며, 별개의 CP 상승 두 분만 CP 학습에서 보류했음. |
| 240044 · SPHC | 후반 `TC-Z3`가 올랐다 내려갈 때 `OP-Z3`가 낮아졌다 높아졌고, 출력이 약 99.37%에서 평평하게 이어졌음. 이 값이 실제 설정 상한인지는 확인되지 않았으며 100% 도달 여부나 큰 변동만으로 LOT 전체를 제외하지 않음. |
| 240058 · DP590 | **정기수리를 위해 소둔로를 멈춘 LOT이며 후반 온도 하강은 정지 과정임.** `CP-Z4`는 가열 온도보다 먼저 낮아졌으므로 따로 보고, 온도 하강 자체를 고장으로 붙이거나 나중의 오일씰·O링 교체만으로 CP 저하 원인을 확정하지 않음. |
| 240061 · SPHC | **정기수리 후 첫 재가동 LOT이며 초기 온도 상승은 재가동 과정임.** 정기수리 종료 후 061 이전에 중간 강종 변경이 없다는 팀 해석을 유지하고, 온도가 오른 뒤에도 높게 남는 `OP-Z4`와 이후 `OP-Z3`의 0 출력·온도 왕복은 따로 검토함. |
| 240069 · SPHC | `CP-Z4`가 여러 차례 짧게 올랐고 가장 큰 상승 때 `TC-Z4`는 오르고 `OP-Z4`는 낮아졌으며, 이후에는 별도의 `OP-Z3` 0 출력과 온도 왕복도 보였음. 기존안은 CP 이탈 22분만 보류했고 나중의 절연 관련 정비·O링 교체를 CP 고장이나 누설의 증거로 확정하지 않았음. |
| 240091 · SPHC | 7개 대상 중 `CP-Z4`의 짧은 이탈과 후반 `OP-Z3` 0 출력·`TC-Z3` 왕복을 각각 검토했음. 기존안은 09/08 23:33 한 분만 CP 학습에서 보류했으며, 출력 0이나 높은 평탄값을 그 자체로 고장 또는 실제 설정 상한이라고 확정하지 않음. |
| 240108 · SPFH590 | 11:27의 낮은 `OP-Z3` 뒤 11:29에 `TC-Z3` 최고가 나타났고, `OP-Z4` 수준은 같은 SPFH590인 024와 비교했음. 제어 반응이나 지연으로 해석할 여지는 있지만 소재 이동시간·원인을 확인한 것은 아니며 기존안에서는 검증 자료로 유지했음. |
| 240114 · SPCEN | 하강 전 `TC-Z4`는 038과 비슷하지만 `OP-Z4`는 낮았고, 후반 가열 온도·출력이 달라져도 `CP-Z4`는 약 0.45를 유지했음. CP까지 낮아진 058과 동일한 사건으로 묶지 않으며, 기존안의 후반 가열 구간 표시는 검증의 이상 기준 계산용이고 전체 평가 자료를 지우라는 뜻이 아님. |
| 240117 · SPCD | 관측 시작부터 온도가 오른 뒤 정기점검·상태기반 정비 작업이 겹쳤고, 오후에는 별도의 CP 저하가 나타났음. 작업이 원인이라고 확정하지 않았으며 SPCD는 이 LOT 하나뿐이고 기존 분할의 학습에는 없으므로 큰 예측 오차를 곧바로 고장으로 해석하지 않음. |
| 240121 · SPCE | 기존 검토에서 `TC-Z3` 최고는 16:42였고 `CP-Z4` 중앙값은 다른 SPCE LOT와 비슷했지만, 주요 추가 검토 대상은 이번 7개 밖의 냉각·과시효 센서였음. 이번 7개에 새 제외 근거를 옮겨 붙이지 않으며 기존안에서는 시험 자료를 유지했음. |
| 240133 · SPCE | 15:54 정비 감지 전후 `CP-Z4` 중앙값은 비슷했고 실제 정비는 온도 관측 종료 뒤인 16:54에 시작했음. 감지 시각 이후를 모두 이상으로 붙이거나 관측하지 않은 수리 효과를 설명하지 않으며 기존안에서는 시험 자료를 유지했음. |

038과 114의 비교값은 각각 해당 LOT를 요약한 사후 비교다. 미래 LOT의 중앙값을 과거 LOT 예측 모델의 기준값으로 사용하지 않는다. 117의 SPCD처럼 비교 LOT가 하나뿐인 강종은 다른 LOT에서의 성능을 확인한 것으로 말하지 않는다.


<details>
<summary>기존 학습 보류 시각과 적용 기준 보기</summary>

아래 시각·행수는 **이전 05의 분당 30개 조건을 적용했을 때의 참고 결과**임. 이번 시각화나 features.csv에는 적용하지 않음.


**학습에서 잠시 빼는 표는 고장 정답표가 아니다.** 원자료는 남기고 안정적으로 운전하는 상태를 배우는 비교 실험에서 특정 센서·시간을 사용하지 않겠다는 선택이다. 보류 표시가 없는 구간도 정상으로 확정된 것이 아니다. 058 정지·061 재가동의 온도 하강·상승 자체를 고장 이상 `1`로 붙이지 않는다.

아래는 기존 05 판단 중 이번 7개에 해당하는 부분만 추린 참고표다. 새 그림을 그렸다고 모델을 다시 학습했거나 팀의 최종 제외 기준이 결정된 것은 아니다. 시간은 **시작·끝이 포함되는 1분 칸**이다. `04:59~05:00`은 `04:59:00 이상 05:01:00 미만`이다.

| LOT | 대상 | 기존안의 보류 시간(2024년) | 뜻 |
| --- | --- | --- | --- |
| 240024 | `CP-Z4` | 03/08 04:59~05:00 | 짧은 CP 저하, 2분 |
| 240037 | `CP-Z4` | 04/13 23:20~23:21, 23:57~23:59; 04/14 03:10~03:11 | 짧은 CP 이탈, 합계 7분 |
| 240038 | `CP-Z4` | 04/16 13:37~13:38 | 짧은 CP 상승, 2분. `OP-Z4` 전체는 유지 |
| 240058 | `CP-Z4` | 06/09 07:33~07:42, 07:43~09:02 | 앞의 이탈과 뒤의 지속 저하를 구분, 합계 90분 |
| 240058 | `TC-Z3`·`OP-Z3`·`TC-Z4`·`OP-Z4`·`TC-Z5`·`OP-Z5` | 06/09 08:14~09:02 | 정기수리를 위한 정지 중 가열 온도 하강, 센서당 49분 |
| 240061 | `TC-Z3`·`OP-Z3`·`TC-Z4`·`OP-Z4`·`TC-Z5`·`OP-Z5` | 06/15 01:04~02:30 | 재가동 중 온도 상승. 87개 시간 칸이며 `OP-Z3` 유효 자료는 84분 |
| 240069 | `CP-Z4` | 07/07 18:59~19:02, 19:08~19:09, 19:11, 19:21~19:22, 19:38~19:39, 19:41~19:42; 07/08 02:30~02:38 | 짧은 CP 이탈, 합계 22분 |
| 240091 | `CP-Z4` | 09/08 23:33 | 짧은 CP 이탈, 1분 |
| 240114 | `TC-Z3`·`OP-Z3`·`TC-Z4`·`OP-Z4`·`TC-Z5`·`OP-Z5` | 11/04 20:10~20:59 | 검증의 이상 기준 계산에서만 보류. 마지막 분은 3관측으로 무효이므로 유효 자료는 센서당 49분 |

- CP의 `0.45±0.05`는 이전 비교 실험의 임시 선택값이지 물리적 정상 범위나 고장 한계가 아니다. 해당 범위 밖 원자료가 들어 있는 1분 칸에서 CP만 보류했으며 앞뒤 5분·다른 센서로 자동 확대하지 않았다.
- 위 가열 시각은 원래 **`TC-Z3~TC-Z6` 네 온도 평균**으로 정한 사후 경계를 보존한 것이다. 새 그림에서 7개 대상만 보여준다고 세 온도 평균으로 새로 계산한 경계가 되는 것은 아니다. 각 출력이 모두 그 시각부터 같은 방향으로 움직였다는 뜻도 아니다.
- 061은 기존 218분 제외를 철회했다. 마지막 30분을 기준으로 한 사후 안정 구간의 시작은 02:31이며, 20분 유지 여부를 확인할 수 있는 02:51과 구분한다.
- 시험 117·121·133의 사건 구간은 추가 삭제하지 않았다. 기존 9·2·3 LOT 분할은 잠정 실험안이며 세 시험 LOT를 이미 살펴봤으므로 아직 보지 않은 최종 시험이라고 부르지 않는다.
- 대상 센서의 표시가 다른 모델의 입력값이나 1분·5분 전 값에 자동 전달되는 것은 아니다. 이전 06은 옛 제외 파일을 읽었으므로 최신 05의 기준으로 얻은 성능처럼 인용하지 않는다.

기존의 과시효·냉각·세정·건조 상세 판단과 그 적용 구간은 이번 7개 비교 밖이다. 그 판단은 원본 보존본과 Git 이력에 남기고, 이번에 새로 잘못된 센서나 제외 대상이라고 결정하지 않는다.

</details>


<details>
<summary>시계의 시각과 경과시간·강종 연결 기준 보기</summary>

240001의 첫 관측은 2024-01-02 03:36임. `13:26`은 오후 1시 26분이며 첫 관측 후 **590분**임. 기존 표의 `OP-Z3 4.30% / TC-Z3 866.84℃`는 13:26:00 이상 13:27:00 미만의 평균임. 13:29의 최고 온도와 다른 시각임. 원자료 한 관측값·1분 평균·1분 중앙값을 구분함.

이번 그림의 점은 분의 시작에 표시됨. 예를 들어 13:26 점의 평균은 13:27이 된 뒤 계산할 수 있음. 과거 그림이 분 가운데에 점을 놓았다면 점 위치도 30초 다를 수 있음.

14개 LOT의 첫 관측은 CR1 강종변경 시각과 일치함. 이벤트에는 8개 강종이 있지만 온도 관측은 7개 강종임. SPCC는 강종변경에는 있고 온도 관측 LOT에는 없음.

6월 수리 중 기록은 6/11 DP590→SPCC, 6/13 SPCC→DP590이고 6/15의 061 시작에 DP590→SPHC임. 058 정지·061 첫 재가동이라는 팀 판단을 유지함. 수리 중 더미 강판이 실제로 통과했는지는 이 기록만으로 확인할 수 없음. 059·060의 온도값을 추정해 채우지 않음.

LOT 사이에 기록이 없는 날짜도 이전 LOT에 붙이는 방식은 사용하지 않음. 첫 관측~마지막 관측 사이와 관측이 없는 기간을 나누어 읽음.

</details>


<details>
<summary>기존 8개 CSV의 시간 연결 결과 보기</summary>

이전 8개 CSV 비교에서 온도 LOT별 첫~마지막 관측 구간의 합은 약 4.12일이었고, 용접 기록이 있는 날짜는 77일이었음. 두 숫자를 실제 조업일수로 해석하지 않음.

온도 관측 기간과 겹친 용접 접합번호는 82개이며 LOT는 240108·240114·240117·240121이었음. 같은 시간에 기록됐다는 뜻이지 같은 소재가 연결되거나 용접이 온도 변화의 원인이라는 뜻은 아님. 소둔로 도착 시각을 확인하려면 소재 추적 정보가 필요함.

소둔로 CAL01 정비의 착수~완료와 온도 관측이 겹친 LOT는 240117이었음. 정비의 감지 시각·계획 수리기간·실제 작업 시각·설비 정지시간을 구분함.

정비는 설비번호 EQP_TAG로 공장을 연결하고, 부품교체는 작업번호 WO_NO로 정비를 연결함. 부품교체에 붙인 정비 착수 시각은 실제 교체 시각이 아님. 설비 설치일은 2024년 온도 관측일이 아니며 부품마스터에는 날짜가 없음.

초기 EDA의 전체 연간 첫~마지막 사이 이벤트 수는 14개 실제 관측 구간에 겹친 기록 수와 다름. 다른 공장 기록을 CR1 기록으로 옮겨 해석하지 않음.

</details>
